# Regression Model Comparison
This notebook compares seven regression models for predicting `TARGET_Second_Path_Range`.


## 1. Data Loading and Evaluation Function
This cell loads the dataset, prepares the train/test split, and defines the dashboard function.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load the cleaned regression dataset
file_path = "uwb_dataset_regression.parquet"
df = pd.read_parquet(file_path)

# Separate features (X) and target (y)
target_col = "TARGET_Second_Path_Range"
if target_col not in df.columns:
    raise ValueError(f"Target column '{target_col}' not found.")

features = [col for col in df.columns if col not in [target_col, 'NLOS']]
X = df[features]
y = df[target_col]

# Split into 80% Training and 20% Testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Dataset loaded. Training on {X_train.shape[0]} rows and {X_train.shape[1]} features.")

# Store each model's results for the final comparison
model_results = []

# Plot model performance and store metrics
def plot_independent_dashboard(model_name, grid, param_name, train_time_sec=None):
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    cv_res = pd.DataFrame(grid.cv_results_)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    model_results.append({
        'Model': model_name,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'Train Time (s)': train_time_sec,
        'Best Params': grid.best_params_
    })

    residuals = y_test - y_pred

    fig, axes = plt.subplots(4, 1, figsize=(8, 20))
    fig.suptitle(f"Model Results: {model_name} (Regression)", fontsize=18, fontweight='bold', y=0.98)

    # Graph 1: Train vs CV RMSE (learning curve)
    param_values = [str(x) for x in cv_res[f'param_{param_name}']]
    train_rmse = -cv_res['mean_train_score']
    test_rmse = -cv_res['mean_test_score']
    axes[0].plot(param_values, train_rmse, marker='s', label='Train RMSE', color='skyblue', lw=2)
    axes[0].plot(param_values, test_rmse, marker='o', label='CV RMSE', color='salmon', lw=2)
    axes[0].set_title("Learning Curve: Train vs. CV RMSE")
    axes[0].set_xlabel(param_name)
    axes[0].set_ylabel("RMSE (lower is better)")
    axes[0].legend()
    axes[0].tick_params(axis='x', rotation=30)

    # Graph 2: Predicted vs Actual (fit quality)
    axes[1].scatter(y_test, y_pred, alpha=0.3, s=10, color='steelblue')
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    axes[1].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
    axes[1].set_title("Predicted vs Actual")
    axes[1].set_xlabel("Actual")
    axes[1].set_ylabel("Predicted")
    axes[1].legend()

    # Graph 3: Residual Distribution (error spread)
    sns.histplot(residuals, bins=40, kde=True, ax=axes[2], color='darkorange')
    axes[2].axvline(0, color='black', linestyle='--', linewidth=1.2)
    axes[2].set_title("Residual Distribution")
    axes[2].set_xlabel("Residual (Actual - Predicted)")

    # Graph 4: Residuals vs Predicted (scatter)
    axes[3].scatter(y_pred, residuals, alpha=0.3, s=10, color='purple')
    axes[3].axhline(0, color='black', linestyle='--', linewidth=1.2)
    axes[3].set_title("Residuals vs Predicted")
    axes[3].set_xlabel("Predicted")
    axes[3].set_ylabel("Residual")

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    print(f"Best Params: {grid.best_params_}")
    print(f"Test RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}")
    if train_time_sec is not None:
        print(f"Training Time: {train_time_sec:.2f} seconds")

## 2. Ridge Regression
This cell tunes Ridge Regression with different regularization strengths.

In [ ]:
# Tune Ridge Regression with different regularization strengths
from sklearn.linear_model import Ridge
param_grid = {'alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000]}
grid = GridSearchCV(
    Ridge(), param_grid, cv=3, return_train_score=True,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
start_time = time.perf_counter()
grid.fit(X_train, y_train)
train_time_sec = time.perf_counter() - start_time
plot_independent_dashboard("Ridge Regression", grid, 'alpha', train_time_sec)

## 3. K-Nearest Neighbors Regression
This cell tunes KNN using different neighborhood sizes.

In [ ]:
# Tune K-Nearest Neighbors using different neighborhood sizes
from sklearn.neighbors import KNeighborsRegressor
param_grid = {'n_neighbors': [3, 5, 7, 9, 11, 15, 21, 31]}
grid = GridSearchCV(
    KNeighborsRegressor(), param_grid, cv=3, return_train_score=True,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
start_time = time.perf_counter()
grid.fit(X_train, y_train)
train_time_sec = time.perf_counter() - start_time
plot_independent_dashboard("K-Nearest Neighbors Regressor", grid, 'n_neighbors', train_time_sec)

## 4. Decision Tree Regression
This cell tunes a Decision Tree by limiting tree depth.

In [ ]:
# Tune a Decision Tree by limiting tree depth
from sklearn.tree import DecisionTreeRegressor
param_grid = {'max_depth': [3, 5, 8, 12, 15, 20, 30, None]}
grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42), param_grid, cv=3, return_train_score=True,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
start_time = time.perf_counter()
grid.fit(X_train, y_train)
train_time_sec = time.perf_counter() - start_time
plot_independent_dashboard("Decision Tree Regressor", grid, 'max_depth', train_time_sec)

## 5. Random Forest Regression
This cell tunes Random Forest using different numbers of trees.

In [ ]:
# Tune Random Forest using different numbers of trees
from sklearn.ensemble import RandomForestRegressor
param_grid = {'n_estimators': [50, 100, 200]}
grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1), param_grid, cv=3, return_train_score=True,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
start_time = time.perf_counter()
grid.fit(X_train, y_train)
train_time_sec = time.perf_counter() - start_time
plot_independent_dashboard("Random Forest Regressor", grid, 'n_estimators', train_time_sec)

## 6. Gradient Boosting Regression
This cell tunes Gradient Boosting using different ensemble sizes.

In [ ]:
# Tune Gradient Boosting using different ensemble sizes
from sklearn.ensemble import GradientBoostingRegressor
param_grid = {'n_estimators': [50, 100, 150, 200, 300, 500]}
grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42), param_grid, cv=3, return_train_score=True,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
start_time = time.perf_counter()
grid.fit(X_train, y_train)
train_time_sec = time.perf_counter() - start_time
plot_independent_dashboard("Gradient Boosting Regressor", grid, 'n_estimators', train_time_sec)

## 7. Support Vector Regression
This cell tunes Support Vector Regression with different C values.

In [ ]:
# Tune Support Vector Regression with different C values
from sklearn.svm import SVR
param_grid = {'C': [0.01, 0.1, 1, 3, 10, 30]}
grid = GridSearchCV(
    SVR(kernel='rbf'), param_grid, cv=3, return_train_score=True,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)  
start_time = time.perf_counter()
grid.fit(X_train, y_train)
train_time_sec = time.perf_counter() - start_time
plot_independent_dashboard("Support Vector Regressor", grid, 'C', train_time_sec)

## 8. Neural Network Regression
This cell tunes the neural network regularization strength.

In [ ]:
# Tune the Neural Network regularization strength
from sklearn.neural_network import MLPRegressor
param_grid = {'alpha': [0.00001, 0.0001, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0]}
grid = GridSearchCV(
    MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42),
    param_grid, cv=3, return_train_score=True,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
start_time = time.perf_counter()
grid.fit(X_train, y_train)
train_time_sec = time.perf_counter() - start_time
plot_independent_dashboard("Neural Network Regressor", grid, 'alpha', train_time_sec)

## 9. Final Summary
This cell compares all tuned models and selects the best one using RMSE, MAE, and R2.

In [ ]:
# Compare all tuned regression models and choose the best one
summary_df = pd.DataFrame(model_results).copy()

# Rank the models using the selection rule:
# 1. Lowest RMSE
# 2. Lowest MAE
# 3. Highest R2 as the tie-break
summary_df = summary_df.sort_values(
    by=['RMSE', 'MAE', 'R2'],
    ascending=[True, True, False]
).reset_index(drop=True)

display(summary_df[['Model', 'RMSE', 'MAE', 'R2', 'Best Params']])

fig, axes = plt.subplots(3, 1, figsize=(14, 18))
fig.suptitle('Final Model Comparison (Selection: RMSE -> MAE -> R2)', fontsize=16, fontweight='bold',y=0.98)

# Graph 1: RMSE comparison
rmse_colors = ['#2ecc71' if v == summary_df['RMSE'].min() else '#3498db' for v in summary_df['RMSE']]
axes[0].bar(summary_df['Model'], summary_df['RMSE'], color=rmse_colors, edgecolor='black')
axes[0].set_title('RMSE by Model (primary criterion, lower is better)')
axes[0].set_ylabel('RMSE')
axes[0].tick_params(axis='x', rotation=30)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# Graph 2: MAE comparison
mae_colors = ['#2ecc71' if v == summary_df['MAE'].min() else '#3498db' for v in summary_df['MAE']]
axes[1].bar(summary_df['Model'], summary_df['MAE'], color=mae_colors, edgecolor='black')
axes[1].set_title('MAE by Model (secondary criterion, lower is better)')
axes[1].set_ylabel('MAE')
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

# Graph 3: R2 comparison
r2_colors = ['#2ecc71' if v == summary_df['R2'].max() else '#3498db' for v in summary_df['R2']]
axes[2].bar(summary_df['Model'], summary_df['R2'], color=r2_colors, edgecolor='black')
axes[2].set_title('R2 by Model (tie-break, higher is better)')
axes[2].set_ylabel('R2')
axes[2].set_xlabel('Model')
axes[2].tick_params(axis='x', rotation=30)
axes[2].grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# The first row is the best model under the chosen rule
best_model = summary_df.iloc[0]
print("Best model by selection rule (RMSE -> MAE -> R2):", best_model['Model'])
print(f"RMSE: {best_model['RMSE']:.4f} | MAE: {best_model['MAE']:.4f} | R2: {best_model['R2']:.4f}")
print("Conclusion: Best model is selected using lowest RMSE first, then MAE, then R2.")